# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`  
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata (name, description)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets, their fields, and corresponding `@id`s. All exploration is indexed by `@id` for unambiguous reference.


In [ ]:
# List all available record sets and their `@id`s

metadata_dict = dataset.metadata.to_json()

# Not all Croissant files include the embedded record sets on the top level.
# Here we extract them explicitly by looking for 'recordSet' in metadata.

record_sets = metadata_dict.get('recordSet', [])

if not record_sets:
    print('No record sets found in the Croissant file! Please verify the schema or meta-data source.')
else:
    print(f"{len(record_sets)} record set(s) found:")
    for idx, rset in enumerate(record_sets):
        if isinstance(rset, dict):
            # Sometimes the entries are dicts with '@id'
            rs_id = rset.get('@id', '--')
            rs_name = rset.get('name', '--')
        else:
            rs_id = rset
            rs_name = '--'
        print(f"  [{idx}] @id: {rs_id}  name: {rs_name}")
    # For demonstration, preview the first record set (if present)
    first_recordset_id = record_sets[0].get('@id') if isinstance(record_sets[0], dict) else record_sets[0]

    # Print the field list for the first record set
    print(f"\nFields for record set @id: {first_recordset_id}")
    record_sets_json = metadata_dict['@graph'] if '@graph' in metadata_dict else []
    if not record_sets_json:
        # Some schemas keep everything in root
        record_sets_json = [metadata_dict]

    target_rs = None
    for obj in record_sets_json:
        if obj.get('@id', None) == first_recordset_id:
            target_rs = obj
            break
    if target_rs is not None and 'field' in target_rs:
        fields = target_rs['field']
        if isinstance(fields, list):
            for f in fields:
                if isinstance(f, dict):
                    print(f"    - @id: {f.get('@id', '--')}, name: {f.get('name', '--')}")
                else:
                    print(f"    - @id: {f}")
        else:
            print(f"    - @id: {fields}")
    else:
        print('  No fields found or @graph record not in schema.')

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. **All references are made by `@id` as per the Croissant schema.**


In [ ]:
# Extract data from each record set

metadata_dict = dataset.metadata.to_json()
record_sets = metadata_dict.get('recordSet', [])
if not record_sets:
    print('No record sets in the Croissant schema -- skipping!')
else:
    # Convert to list of @id strings for user
    def resolve_rs_id(rset):
        if isinstance(rset, dict):
            return rset.get('@id', None)
        else:
            return rset
    record_set_ids = [resolve_rs_id(x) for x in record_sets]

    dataframes = {}
    for rs_id in record_set_ids:
        print(f"Loading record set: {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[rs_id] = df
                print(f"  Loaded shape: {df.shape}, columns: {df.columns.tolist()}")
            else:
                print(f"  No records found for {rs_id}")
        except Exception as exc:
            print(f"  Exception for {rs_id}: {exc}")

    # For demonstration, show columns and first rows of the first dataframe
    if record_set_ids and record_set_ids[0] in dataframes:
        first_df = dataframes[record_set_ids[0]]
        print("\nFirst 5 rows for record set", record_set_ids[0])
        print(first_df.head())
    else:
        print('No DataFrames loaded!')

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All column accesses use the Croissant field `@id` string from the schema overview, not display names.


In [ ]:
# Example: Numeric field filtering and normalization

# Edit these variables to match the desired record set and field @ids from above
# Example placeholder values -- replace with actual @id values found in section 2 output

record_set_id = None
numeric_field_id = None
group_field_id = None

# Try to auto-detect a usable example if user did not edit variables manually
if not record_set_id and len(dataframes) > 0:
    record_set_id = next(iter(dataframes.keys()))
    print(f"Using record_set_id: {record_set_id}")

df = dataframes.get(record_set_id, None)
if df is None:
    print("No dataframe for EDA!")
else:
    # Try to auto-detect a numeric field (float/integer columns)
    candidates = df.select_dtypes(include=['number']).columns.tolist()
    if not candidates:
        print("No numeric columns found in the record set.")
    else:
        if numeric_field_id is None:
            numeric_field_id = candidates[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Filter: greater than threshold
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 1
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt a grouping field: select any non-numeric column for example
        non_numeric = df.select_dtypes(exclude=['number']).columns.tolist()
        if not non_numeric:
            print("No groupable field (categorical) found.")
        else:
            if group_field_id is None:
                group_field_id = non_numeric[0]
            print(f"Grouping by {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())

## 5. Visualization

Visualize the distribution or relationships of fields in the dataset. Uses only field `@id` as column names.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize histogram for the detected numeric field

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to:
- Load and inspect metadata about a mass spectrometry dataset using the Croissant schema.
- List all available record sets and fields referenced by their `@id` for reproducibility.
- Load and explore the records as Pandas DataFrames, including basic filtering and normalization operations by `@id`.
- Visualize the distribution of numeric fields from the data.

To further analyze this dataset, repeat steps 2–5 for other record sets and fields, as appropriate for your research question.
